#Data Pipeline using LLM

Go to https://groq.com/ and generate a Free API Key.


1. Data Cleaning:

  Begin by loading the dataset into your Colab environment.
  Use pandas functions like head(), info(), describe(), and value_counts() to explore the structure, data types, and basic statistics of the dataset.

  Identify potential data quality issues such as missing values, inconsistent formats, or incorrect entries.
  Prompt Engineering:

  This is the core of the lab. Your task is to craft a prompt that instructs an LLM (Groq's LLama2) to clean the data.

  The Cleaning Goals: Your prompt should guide the LLM to perform the following tasks:

  * Address missing values: Infer or fill in missing information where possible (e.g., city names from addresses).
  * Standardize text: Correct spelling, apply consistent capitalization, and ensure uniformity in categorical values.
  * Validate and format: Ensure that addresses are in a standard format (e.g., "Street, Borough, NY"), and that dates and times follow ISO 8601.
  * Categorize: Assign clear categories to ambiguous complaint descriptions (e.g., "Noise," "Non-Noise").

  You are not given the prompt used in the example code, but you are given the expected results.
  Iterative Refinement: Start with a basic prompt and gradually refine it based on the LLM's output. Observe how the LLM responds and make adjustments to improve the cleaning process.

2. Data Validation:

  After cleaning the data, write unit tests (using Python's assert statements) to validate the output.
  Your tests should check data types, value ranges, and ensure that required fields are not null.
  Generate code for tests. Try to see the problems in running the code.

Submission: Write your prompts in a text file and upload on LMS.

In [1]:
# Groq-Powered Data Engineering Pipeline

# Step 1: Install Required Libraries
!pip install groq itables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.7/126.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.3 MB/s eta 0:00:00


In [2]:
# Step 2: Import Libraries
from groq import Groq
import pandas as pd
from itables import init_notebook_mode
from google.colab import userdata
import json
import re
from tqdm import tqdm
import itables

init_notebook_mode(all_interactive=True)

In [3]:
# Load a manageable sample (500 rows) for this lab
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=100"
df = pd.read_csv(url)
df

unique_key             created_date              closed_date agency  \
0     64677480  2025-04-18T01:35:22.000                      NaN   NYPD   
1     64672937  2025-04-18T01:34:18.000                      NaN   NYPD   
2     64679960  2025-04-18T01:33:57.000                      NaN   NYPD   
3     64675055  2025-04-18T01:33:38.000                      NaN   NYPD   
4     64680154  2025-04-18T01:33:25.000                      NaN   NYPD   
..         ...                      ...                      ...    ...   
95    64676315  2025-04-18T00:59:11.000                      NaN   NYPD   
96    64676287  2025-04-18T00:58:40.000                      NaN   NYPD   
97    64673882  2025-04-18T00:58:28.000                      NaN   NYPD   
98    64674754  2025-04-18T00:58:25.000  2025-04-18T01:26:21.000   NYPD   
99    64673912  2025-04-18T00:58:18.000                      NaN   NYPD   

                        agency_name           complaint_type  \
0   New York City Police Department          Illegal Parking   
1   New York City Police Department  Noise - Street/Sidewalk   
2   New York City Police Department  Noise - Street/Sidewalk   
3   New York City Police Department      Noise - Residential   
4   New York City Police Department      Noise - Residential   
..                              ...                      ...   
95  New York City Police Department      Noise - Residential   
96  New York City Police Department       Noise - Commercial   
97  New York City Police Department          Illegal Parking   
98  New York City Police Department  Noise - Street/Sidewalk   
99  New York City Police Department         Blocked Driveway   

                        descriptor               location_type  incident_zip  \
0   Double Parked Blocking Vehicle             Street/Sidewalk         10463   
1                 Loud Music/Party             Street/Sidewalk         10464   
2                 Loud Music/Party             Street/Sidewalk         10035   
3                 Banging/Pounding  Residential Building/House         11226   
4                 Loud Music/Party  Residential Building/House         10467   
..                             ...                         ...           ...   
95                 Loud Television  Residential Building/House         11375   
96                Loud Music/Party         Club/Bar/Restaurant         11206   
97          License Plate Obscured             Street/Sidewalk         11422   
98                Loud Music/Party             Street/Sidewalk         11217   
99                  Partial Access             Street/Sidewalk         11418   

          incident_address  ... vehicle_type taxi_company_borough  \
0         3435 GILES PLACE  ...          Van                  NaN   
1       219 FORDHAM STREET  ...          NaN                  NaN   
2     523 EAST  117 STREET  ...          NaN                  NaN   
3     370 EAST   31 STREET  ...          NaN                  NaN   
4   3505 ROCHAMBEAU AVENUE  ...          NaN                  NaN   
..                     ...  ...          ...                  ...   
95          102-45 62 ROAD  ...          NaN                  NaN   
96        29 LOCUST STREET  ...          NaN                  NaN   
97        241-25 148 DRIVE  ...          NaN                  NaN   
98    86 FORT GREENE PLACE  ...          NaN                  NaN   
99         102-39 85 DRIVE  ...          NaN                  NaN   

   taxi_pick_up_location bridge_highway_name bridge_highway_direction  \
0                    NaN                 NaN                      NaN   
1                    NaN                 NaN                      NaN   
2                    NaN                 NaN                      NaN   
3                    NaN                 NaN                      NaN   
4                    NaN                 NaN                      NaN   
..                   ...                 ...                      ...   
95                   NaN         

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 41 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   unique_key                      100 non-null    int64  
 1   created_date                    100 non-null    object 
 2   closed_date                     14 non-null     object 
 3   agency                          100 non-null    object 
 4   agency_name                     100 non-null    object 
 5   complaint_type                  100 non-null    object 
 6   descriptor                      99 non-null     object 
 7   location_type                   94 non-null     object 
 8   incident_zip                    100 non-null    int64  
 9   incident_address                98 non-null     object 
 10  street_name                     98 non-null     object 
 11  cross_street_1                  98 non-null     object 
 12  cross_street_2                  96 no

In [5]:
client = Groq(api_key=userdata.get("YOUR_GROQ_API_KEY"))

In [8]:
sample_df = df.head(10)  # Start with 10 rows due to complexity & API limits

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
  print(row)
  print("-----------------------")
  print(row.to_dict())
  break


  0%|          | 0/10 [00:00<?, ?it/s]

unique_key                                                              64677480
created_date                                             2025-04-18T01:35:22.000
closed_date                                                                  NaN
agency                                                                      NYPD
agency_name                                      New York City Police Department
complaint_type                                                   Illegal Parking
descriptor                                        Double Parked Blocking Vehicle
location_type                                                    Street/Sidewalk
incident_zip                                                               10463
incident_address                                                3435 GILES PLACE
street_name                                                          GILES PLACE
cross_street_1                                                      CANNON PLACE
cross_street_2              

In [33]:
def llm_complex_clean(record):
    prompt = f"""

    You are an expert data engineer who excels in cleaning and preprocessing datasets.

    Given the following record:

    {record.to_dict()}

    I want you to PERFORM the following tasks efficiently:

    - **Address missing values** by inferring or filling in missing information where possible (examples: city names from addresses, bridge names from city, closing date must be ahead of opening date, taxi location or borough from nearby boroughs, facility type from location type, vehicle type from borough or incident ).
    - **Standardize text**: Correct spelling, apply consistent capitalization, and ensure uniformity in categorical values.
    - **Validate and format**: Ensure that addresses are in a standard format (example: "Street, Borough, NY"), and that dates and times follow ISO 8601.
    - **Categorize**: Assign clear categories to ambiguous complaint descriptions (example: "Noise," "Non-Noise").

    I want you to AVOID the following:

    - Creating new columns or changing column names
    - DO NOT add any comments or unwanted signs like # or * except the cleaned values in the record
    - **NO VALUE should be 'Unspecified' or an empty string**,you MUST infer it from the information in the record, **mentioning it as null / nan is the last resort**
    - If one text field is ALL CAPS then all text fields in that record should be ALL CAPS, **ensure consistent textual formats like capitalize consistently**.
    - For categorical values, NO INCONSISTENCY will be tolerated, they should be uniform.
    - AVOID nonsensical inferrences, for example if status is 'in progress', closed date and resolution description should be left as Null.

    """

    chat_completion = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    result = chat_completion.choices[0].message.content.strip()

    return result


In [13]:
# Feel Free to define any number of functions.

def extract_dict_from_response(response_string):
    """
    Extracts a dictionary from a string using regular expressions and fixes JSON formatting.

    Args:
    response_string: The string containing the dictionary representation.

    Returns:
    A dictionary extracted from the response string.
    """
    # Define a regular expression pattern to match the dictionary structure
    pattern = r"\{.*?\}"  # Matches any characters between curly braces

    # Find all matches in the response string
    matches = re.findall(pattern, response_string, re.DOTALL)

    # If matches are found, extract the first match and fix JSON formatting
    if matches:
        try:
            # Replace single quotes with double quotes for keys and values
            json_string = matches[0].replace("'", '"')
            # Replace Python's None with JSON's null
            json_string = json_string.replace("None", "null")

            # Parse the fixed JSON string
            extracted_dict = json.loads(json_string)
            return extracted_dict
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            return None  # Or raise an exception if desired
    else:
        print("No dictionary structure found in the response.")
        return None

In [36]:
cleaned_records = []
sample_df = df.head(10)  # Start with 10 rows due to complexity & API limits

i=0
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    try:
        print(row.to_dict())
        cleaned_record = llm_complex_clean(row)
        cleaned_record = extract_dict_from_response(cleaned_record)
        cleaned_records.append(cleaned_record)
        i+=1
        if i==8:
          break
    except Exception as e:
        print(f"Error cleaning row {_}: {e}")

cleaned_df = pd.DataFrame(cleaned_records)
cleaned_df.head()

  0%|          | 0/10 [00:00<?, ?it/s]

{'unique_key': 64677480, 'created_date': '2025-04-18T01:35:22.000', 'closed_date': nan, 'agency': 'NYPD', 'agency_name': 'New York City Police Department', 'complaint_type': 'Illegal Parking', 'descriptor': 'Double Parked Blocking Vehicle', 'location_type': 'Street/Sidewalk', 'incident_zip': 10463, 'incident_address': '3435 GILES PLACE', 'street_name': 'GILES PLACE', 'cross_street_1': 'CANNON PLACE', 'cross_street_2': 'SEDGWICK AVENUE', 'intersection_street_1': 'CANNON PLACE', 'intersection_street_2': 'SEDGWICK AVENUE', 'address_type': 'ADDRESS', 'city': 'BRONX', 'landmark': 'GILES PLACE', 'facility_type': nan, 'status': 'In Progress', 'due_date': nan, 'resolution_description': nan, 'resolution_action_updated_date': nan, 'community_board': '08 BRONX', 'bbl': 2032580228.0, 'borough': 'BRONX', 'x_coordinate_state_plane': 1012598.0, 'y_coordinate_state_plane': 260247.0, 'open_data_channel_type': 'PHONE', 'park_facility_name': 'Unspecified', 'park_borough': 'BRONX', 'vehicle_type': 'Van', 

 10%|█         | 1/10 [00:00<00:08,  1.04it/s]

{'unique_key': 64672937, 'created_date': '2025-04-18T01:34:18.000', 'closed_date': nan, 'agency': 'NYPD', 'agency_name': 'New York City Police Department', 'complaint_type': 'Noise - Street/Sidewalk', 'descriptor': 'Loud Music/Party', 'location_type': 'Street/Sidewalk', 'incident_zip': 10464, 'incident_address': '219 FORDHAM STREET', 'street_name': 'FORDHAM STREET', 'cross_street_1': 'FORDHAM PLACE', 'cross_street_2': 'FORDHAM PLACE', 'intersection_street_1': 'FORDHAM PLACE', 'intersection_street_2': 'FORDHAM PLACE', 'address_type': 'ADDRESS', 'city': 'BRONX', 'landmark': 'FORDHAM STREET', 'facility_type': nan, 'status': 'In Progress', 'due_date': nan, 'resolution_description': nan, 'resolution_action_updated_date': nan, 'community_board': '10 BRONX', 'bbl': 2056440258.0, 'borough': 'BRONX', 'x_coordinate_state_plane': 1044134.0, 'y_coordinate_state_plane': 248271.0, 'open_data_channel_type': 'ONLINE', 'park_facility_name': 'Unspecified', 'park_borough': 'BRONX', 'vehicle_type': nan, '

 20%|██        | 2/10 [00:01<00:07,  1.13it/s]

...

{'unique_key': 64681084, 'created_date': '2025-04-18T01:33:22.000', 'closed_date': nan, 'agency': 'NYPD', 'agency_name': 'New York City Police Department', 'complaint_type': 'Noise - Residential', 'descriptor': 'Loud Talking', 'location_type': 'Residential Building/House', 'incident_zip': 11239, 'incident_address': '135 ELMIRA LOOP', 'street_name': 'ELMIRA LOOP', 'cross_street_1': 'SCHROEDERS AVENUE', 'cross_street_2': 'BEND', 'intersection_street_1': 'SCHROEDERS AVENUE', 'intersection_street_2': 'BEND', 'address_type': 'ADDRESS', 'city': 'BROOKLYN', 'landmark': 'ELMIRA LOOP', 'facility_type': nan, 'status': 'In Progress', 'due_date': nan, 'resolution_description': nan, 'resolution_action_updated_date': nan, 'community_board': '05 BROOKLYN', 'bbl': 3044520085.0, 'borough': 'BROOKLYN', 'x_coordinate_state_plane': 1017337.0, 'y_coordinate_state_plane': 175737.0, 'open_data_channel_type': 'PHONE', 'park_facility_name': 'Unspecified', 'park_borough': 'BROOKLYN', 'vehicle_type': nan, 'taxi_

 70%|███████   | 7/10 [01:05<00:38, 12.83s/it]

{'unique_key': 64677685, 'created_date': '2025-04-18T01:32:48.000', 'closed_date': nan, 'agency': 'NYPD', 'agency_name': 'New York City Police Department', 'complaint_type': 'Noise - Street/Sidewalk', 'descriptor': 'Loud Music/Party', 'location_type': 'Street/Sidewalk', 'incident_zip': 10464, 'incident_address': '219 FORDHAM STREET', 'street_name': 'FORDHAM STREET', 'cross_street_1': 'FORDHAM PLACE', 'cross_street_2': 'FORDHAM PLACE', 'intersection_street_1': 'FORDHAM PLACE', 'intersection_street_2': 'FORDHAM PLACE', 'address_type': 'ADDRESS', 'city': 'BRONX', 'landmark': 'FORDHAM STREET', 'facility_type': nan, 'status': 'In Progress', 'due_date': nan, 'resolution_description': nan, 'resolution_action_updated_date': nan, 'community_board': '10 BRONX', 'bbl': 2056440258.0, 'borough': 'BRONX', 'x_coordinate_state_plane': 1044134.0, 'y_coordinate_state_plane': 248271.0, 'open_data_channel_type': 'MOBILE', 'park_facility_name': 'Unspecified', 'park_borough': 'BRONX', 'vehicle_type': nan, '

 70%|███████   | 7/10 [01:21<00:34, 11.62s/it]


unique_key             created_date              closed_date agency  \
0    64677480  2025-04-18T01:35:22.000  2025-04-18T02:35:22.000   NYPD   
1    64672937  2025-04-18T01:34:18.000  2025-04-20T00:00:00.000   NYPD   
2    64679960  2025-04-18T01:33:57.000  2025-04-18T02:21:19.000   NYPD   
3    64675055  2025-04-18T01:33:38.000  2025-04-20T00:00:00.000   NYPD   
4    64680154  2025-04-18T01:33:25.000  2025-04-20T00:00:00.000   NYPD   

                       agency_name   complaint_type  \
0  New York City Police Department  Illegal Parking   
1  New York City Police Department            Noise   
2  New York City Police Department            Noise   
3  New York City Police Department            Noise   
4  New York City Police Department            Noise   

                       descriptor               location_type  incident_zip  \
0  Double Parked Blocking Vehicle             Street/Sidewalk         10463   
1                Loud Music/Party             Street/Sidewalk         10464   
2                Loud Music/Party             Street/Sidewalk         10035   
3                Banging/Pounding  Residential Building/House         11226   
4                Loud Music/Party  Residential Building/House         10467   

                    incident_address  ... vehicle_type taxi_company_borough  \
0        3435 GILES PLACE, BRONX, NY  ...          Van                BRONX   
1      219 FORDHAM STREET, BRONX, NY  ...          Car                BRONX   
2     EAST 117 STREET, MANHATTAN, NY  ...          Car            MANHATTAN   
3            31 STREET, BROOKLYN, NY  ...          Car             BROOKLYN   
4  3505 ROCHAMBEAU AVENUE, BRONX, NY  ...          Car                BRONX   

     taxi_pick_up_location bridge_highway_name bridge_highway_direction  \
0              GILES PLACE     SEDGWICK AVENUE                    North   
1           FORDHAM STREET        FORDHAM ROAD                    North   
2          EAST 117 STREET      UNNAMED BRIDGE                    NORTH   
3  31 STREET, BROOKLYN, NY     Brooklyn Bridge                     East   
4       EAST GUN HILL ROAD   ROCHAMBEAU AVENUE                     East   

            road_ramp bridge_highway_segment   latitude  longitude  \
0        CANNON PLACE        SEDGWICK AVENUE  40.880947 -73.897486   
1      FORDHAM STREET           FORDHAM ROAD  40.847919 -73.783551   
2        UNNAMED ROAD         UNNAMED BRIDGE  40.795612 -73.932358   
3     Brooklyn Bridge        Brooklyn Bridge  40.641809 -73.947640   
4  EAST GUN HILL ROAD      ROCHAMBEAU AVENUE  40.881917 -73.879226   

                                  location  
0  (40.88094656085681, -73.89748603249454)  
1  (40.84791852851222, -73.78355075766268)  
2  (40.79561177892521, -73.93235774350866)  
3   (40.64180891484026, -73.9476403447945)  
4  (40.88191698583422, -73.87922572142527)  

[5 rows x 41 columns]

Data Validation

In [81]:
def generate_complex_validation_tests(record):
    prompt = f"""
    THERE SHOULD BE NO INSTANCE OF ` IN THE FINAL OUTPUT

    You are an expert data engineer who excels in writing code for unit tests on cleaned datasets for validation.

    Given the cleaned NYC 311 data record below:

    {record}



    **DO NOT include backticks like ` to format the code as markdown. DO NOT include any other text in output.**
    example: print("hello world") AND NOT  ``` print("hello world") ```
    Write validation tests for the conditions mentioned below and run them.
    Ensure the final output is ONLY the executable code / main program. In your final output, you should also run the defined functions.

    - All dates must be in valid ISO 8601 datetime format like YYYY-MM-DDTHH:MM:SS
    - unique_key column must be unique and non null integer
    - incident_zip should be 5 digit integer only.
    - No value should be empty string
    - the tuple in location should match the corresponding (latitude, longitude) values
    - if status shows 'in progress' then closed date, resolution_description MUST be Null / None.

      Avoid the following things:
    - Do NOT write any text other than the code in your final output(**I will be running the tests automatically NOT manually**).
    - Do NOT include any comments # or asterisks * in the final output.
    """

    chat_completion = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return chat_completion.choices[0].message.content.strip()

# Generate tests based on first cleaned record
test_code = generate_complex_validation_tests(df.iloc[0].to_dict())
print(test_code)

print("Testing the cleaned NYC 311 data record")

import pandas as pd
import numpy as np
from datetime import datetime

def validate_dates(df):
    for index, row in df.iterrows():
        if not isinstance(row['created_date'], str):
            print("Error: created_date is not a string")
            return False
        try:
            datetime.strptime(row['created_date'], '%Y-%m-%dT%H:%M:%S')
        except ValueError:
            print("Error: created_date is not in valid ISO 8601 datetime format")
            return False
        if not isinstance(row['closed_date'], type(None)):
            try:
                datetime.strptime(row['closed_date'], '%Y-%m-%dT%H:%M:%S')
            except ValueError:
                print("Error: closed_date is not in valid ISO 8601 datetime format")
                return False
    return True

def validate_unique_key(df):
    if df['unique_key'].dtype != np.int64:
        print("Error: unique_key is not an integer")
        return False
    if

In [82]:
# Evaluate tests programmatically (OPTIONAL)
exec(test_code)

Testing the cleaned NYC 311 data record
Error: created_date is not in valid ISO 8601 datetime format
False
True
True
Error: There are empty strings in the dataset
False
True
Error: If status is 'In Progress', then closed_date and resolution_description must be Null/None
False
